<h1>Mindestanforderungen 6 - Klassifikation</h1>
<h1>Mindestanforderungen 6</h1>

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.datasets import fetch_20newsgroups

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC

from sklearn.pipeline import Pipeline

from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict

<h2>Daten einlesen</h2>

In [ ]:
df = pd.read_csv("stack-overflow-developer-survey-2025/survey_results_public_without_SO.csv")

<h2>Daten aufteilen in Train und Test Split</h3>

In [ ]:
#y = df["JobSat"].fillna("Unknown").astype(str)
df = df.dropna(subset=["JobSat"])

job_sat_num = df["JobSat"].astype(float)

def map_job_sat(x):
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"

y = job_sat_num.apply(map_job_sat)


text_cols = df.select_dtypes(include=["object"]).columns.tolist()
X = df[text_cols].fillna("").agg(" ".join, axis=1)

print("X shape:", getattr(X, "shape", None), "len(X):", len(X))
print("y shape:", getattr(y, "shape", None), "len(y):", len(y))


# Train / Temp Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_df=0.5,
        analyzer="word"
    )),
    ("classifier", LinearSVC())
])

pipeline

<h2>GridSearchCV, um optimale Parameter herauszufinden</h2>

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'vectorizer__analyzer': ('word', 'char'),
    'classifier__C': (0.1, 1, 10)
}

gs = GridSearchCV(pipeline, parameters, cv=3, n_jobs=-1)
gs.fit(X_train, y_train)

print("Best Score:", gs.best_score_)
print("Best Params:", gs.best_params_)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred_test = pipeline.predict(X_test)
print(classification_report(y_test, y_pred_test))

In [ ]:
vectorizer = TfidfVectorizer()
feature_selection = SelectFromModel (LinearSVC(penalty="l1", dual=False))
classifier = LinearSVC()

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("feature_selection", feature_selection),
    ("classifier", classifier)
])

parameters = {
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'vectorizer__analyzer': ('word', 'char'),
    'feature_selection__threshold': (None, 'mean')
    #'classifier__kernel': ('linear', 'rbf')
}

grid_search = GridSearchCV(pipeline, param_grid=parameters, verbose=10)

grid_search.fit(X_train, y_train)

In [ ]:
print(grid_search.best_estimator_)

In [ ]:
final_pipeline = grid_search.best_estimator_

final_pipeline.fit(X_train, y_train)
print("Default Score des Klassifizieres: Accuracy=", final_pipeline.score(X_test, y_test))

test_labels = final_pipeline.predict(X_test)
print(classification_report(y_test, test_labels))